# Multiclass: lesion segmentation, visual comparison of masks

Two preprocessing branches:

1. **Feature branch** (`preprocessing/preprocessing.py`): resize, hair removal (black-hat), then Shades of Gray. There is no smoothing, so the texture is preserved. All features are computed on this image.
2. **Mask branch** (`preprocessing/segmentation.py`): starts from the feature-branch image, applies strong smoothing, then Otsu or k-means, then post-processing (morphology, hole filling, keeping the central component). This branch is only used to get the lesion mask.

The notebook compares several segmentation methods visually, and on a larger sample it compares the mask area and how often each method fails (fallback).

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing import preprocess, resize_image, segment_lesion, region_masks, overlay_mask

SEED = 42

## 0. Configuration

In [ ]:
TASK = "Multiclass"
DATA_DIR = PROJECT_ROOT / TASK
SPLIT = "train"

SHORT_SIDE = 256
FEATURE_PREPROC = dict(short_side=SHORT_SIDE, hair=True, color_constancy=True, normalize_brightness=False)

# segmentation methods to compare (all use median smoothing, ksize=11)
METHODS = {
    "Otsu (blue)":   dict(method="otsu", channel="blue"),
    "Otsu (L*)":     dict(method="otsu", channel="L"),
    "Otsu (PCA)":    dict(method="otsu", channel="pca"),
    "k-means k=2":   dict(method="kmeans", k=2),
    "k-means k=3":   dict(method="kmeans", k=3, kmeans_lesion="darkest"),
    "k-means k=3 (2 darkest)": dict(method="kmeans", k=3, kmeans_lesion="all_but_brightest"),
}
SMOOTH = dict(smooth="median", smooth_ksize=11)

N_SHOW = 4        # images per class in the visual grid
N_STATS = 150     # images per class for the area / fallback statistics

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
split_dir = next(p for p in DATA_DIR.iterdir() if p.is_dir() and p.name.lower().startswith(SPLIT))
df = pd.DataFrame([{"path": f, "class": d.name.lower()}
                   for d in sorted(p for p in split_dir.iterdir() if p.is_dir())
                   for f in sorted(d.rglob("*")) if f.suffix.lower() in IMG_EXT])
CLASSES = sorted(df["class"].unique())
df["class"].value_counts()

## 1. Feature-branch preprocessing

This compares the raw image with the output of the two Shades of Gray options: tint only (the default), and tint plus brightness.

In [ ]:
paths = df.groupby("class", group_keys=False).sample(1, random_state=SEED)["path"].tolist()
cols = ["raw (resized)", "hair + SoG (tint)", "hair + SoG (tint + brightness)"]
fig, axes = plt.subplots(len(paths), 3, figsize=(10, 3.1 * len(paths)))
axes = np.atleast_2d(axes)
for r, p in enumerate(paths):
    raw = cv2.imread(str(p))
    ims = [resize_image(raw, SHORT_SIDE),
           preprocess(raw, **FEATURE_PREPROC),
           preprocess(raw, **{**FEATURE_PREPROC, "normalize_brightness": True})]
    for c, im in enumerate(ims):
        axes[r, c].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(cols[c], fontsize=10)
    axes[r, 0].text(-0.05, 0.5, df.loc[df.path == p, "class"].iloc[0], transform=axes[r, 0].transAxes,
                    rotation=90, ha="right", va="center", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Masks: visual comparison

For each class the grid shows `N_SHOW` random images, one column per method. The title of each panel gives the mask area as a percentage of the image. A **red** contour and ⚠ mean the sanity check failed and the central-ellipse fallback was used.

In [ ]:
def show_masks(cls, n=N_SHOW, seed=SEED):
    paths = df[df["class"] == cls]["path"].sample(min(n, (df["class"] == cls).sum()), random_state=seed)
    ncol = 1 + len(METHODS)
    fig, axes = plt.subplots(len(paths), ncol, figsize=(2.6 * ncol, 2.5 * len(paths)))
    axes = np.atleast_2d(axes)
    for r, p in enumerate(paths):
        img = preprocess(cv2.imread(str(p)), **FEATURE_PREPROC)
        axes[r, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        for c, (name, kw) in enumerate(METHODS.items(), start=1):
            mask, info = segment_lesion(img, **kw, **SMOOTH)
            col = (255, 0, 0) if info["fallback"] else (0, 255, 0)
            axes[r, c].imshow(overlay_mask(img, mask, color=col))
            flag = " ⚠" if info["fallback"] else ""
            axes[r, c].set_title(f"{name}\n{info['area_frac']*100:.0f}%{flag}" if r == 0
                                 else f"{info['area_frac']*100:.0f}%{flag}", fontsize=8)
        if r == 0: axes[r, 0].set_title("preprocessed", fontsize=8)
    for ax in axes.ravel(): ax.axis("off")
    fig.suptitle(f"{TASK} / {cls}", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

for cls in CLASSES:
    show_masks(cls)

## 3. Spatial-context regions

Once there is a mask, `region_masks()` gives the **lesion**, its **inner** core, the **border** ring and the surrounding **skin**. Features computed on each region, and the lesion − skin differences, add spatial context.

In [ ]:
p = df.sample(1, random_state=SEED + 1)["path"].iloc[0]
img = preprocess(cv2.imread(str(p)), **FEATURE_PREPROC)
mask, info = segment_lesion(img, **METHODS["Otsu (blue)"], **SMOOTH)
reg = region_masks(mask)
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title("image")
for ax, (name, m) in zip(axes[1:], reg.items()):
    vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy(); vis[m == 0] = (vis[m == 0] * 0.15).astype(np.uint8)
    ax.imshow(vis); ax.set_title(name)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Statistics on a larger sample

For `N_STATS` images per class this reports, for each method:

- the **fallback rate**: how often the mask was implausible (<2% or >80% of the image);
- the **mask area** distribution;
- the **time** per image.

A good method has a low fallback rate in every class. For Multiclass, look closely at BCC and SCC, which tend to be less pigmented.

In [ ]:
sample = df.sample(frac=1, random_state=SEED).groupby("class").head(N_STATS).reset_index(drop=True)

rows = []
t_method = {m: 0.0 for m in METHODS}
for p, cls in zip(sample["path"], sample["class"]):
    img = preprocess(cv2.imread(str(p)), **FEATURE_PREPROC)
    for name, kw in METHODS.items():
        t0 = time.time()
        _, info = segment_lesion(img, **kw, **SMOOTH)
        t_method[name] += time.time() - t0
        rows.append({"class": cls, "method": name, "area_frac": info["area_frac"], "fallback": info["fallback"]})
stats = pd.DataFrame(rows)

fallback = stats.pivot_table(index="method", columns="class", values="fallback", aggfunc="mean").mul(100).round(1)
fallback["ALL"] = stats.groupby("method")["fallback"].mean().mul(100).round(1)
fallback["ms / image"] = pd.Series({m: 1000 * t / len(sample) for m, t in t_method.items()}).round(1)
print("Fallback rate (%) and time per image")
fallback.loc[list(METHODS)]

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5.5 * len(CLASSES), 4), sharey=True)
for ax, cls in zip(np.atleast_1d(axes), CLASSES):
    s = stats[stats["class"] == cls]
    ax.boxplot([s[s.method == m]["area_frac"] * 100 for m in METHODS], showfliers=False)
    ax.set_xticks(range(1, len(METHODS) + 1), list(METHODS), rotation=30, ha="right", fontsize=8)
    ax.axhspan(2, 80, color="green", alpha=0.06)
    ax.set_title(cls); ax.set_ylabel("mask area (% of image)")
fig.suptitle("Mask area per method (green band = accepted range)", y=1.02)
plt.tight_layout(); plt.show()

## 5. Worst cases

These are the images where the chosen method fell back. They are useful for tuning `min_frac`/`max_frac`, the smoothing, or the channel, and they make a good figure for the report.

In [ ]:
CHOSEN = "Otsu (blue)"   # change after looking at the table above

fails = stats[(stats.method == CHOSEN) & stats.fallback].index
fail_paths = [sample.iloc[i // len(METHODS)]["path"] for i in fails][:8]
if not fail_paths:
    print(f"No fallbacks for {CHOSEN} in this sample")
else:
    fig, axes = plt.subplots(1, len(fail_paths), figsize=(2.8 * len(fail_paths), 3))
    for ax, p in zip(np.atleast_1d(axes), fail_paths):
        img = preprocess(cv2.imread(str(p)), **FEATURE_PREPROC)
        m, info = segment_lesion(img, **METHODS[CHOSEN], **SMOOTH, use_fallback=False)
        ax.imshow(overlay_mask(img, m, color=(255, 0, 0))); ax.axis("off")
        ax.set_title(f"{p.parent.name}  {info['area_frac']*100:.0f}%", fontsize=8)
    fig.suptitle(f"Fallback cases: {CHOSEN} (raw mask before fallback)")
    plt.tight_layout(); plt.show()

## 6. Notes

*(Fill in after running.)*

- Best method per class (visual + fallback rate):
- Typical failure modes (light lesions, lesion filling the image, dark skin folds...):
- Chosen configuration for feature extraction: